In [0]:
--Checking user profile records
SELECT
    COUNT(*) AS total_records,
    COUNT(DISTINCT UserID) AS unique_users
FROM retail.default.userprofiles;

--Checking viewership profile records
SELECT
    COUNT(*) AS total_viewership_records,
    COUNT(DISTINCT COALESCE(UserID0, UserID4)) AS unique_viewers
FROM retail.default.viewership;

--Checking missing Regions
SELECT
    COUNT(*) AS missing_region
FROM retail.default.userprofiles
WHERE Province IS NULL
   OR TRIM(Province) = ''
   OR Province = 'None';

WITH user_profiles AS (

    SELECT
        UserID,

        CASE
            WHEN Province IS NULL
              OR TRIM(Province) = ''
              OR Province = 'None'
            THEN 'Uncategorized'
            ELSE Province
        END AS Region,

        age,

        CASE
            WHEN age = 0 THEN 'Infants'
            WHEN age BETWEEN 1 AND 12 THEN 'Kids'
            WHEN age BETWEEN 13 AND 19 THEN 'Teenager'
            WHEN age BETWEEN 20 AND 35 THEN 'Youth'
            WHEN age BETWEEN 36 AND 50 THEN 'Adult'
            WHEN age BETWEEN 51 AND 65 THEN 'Elder'
            WHEN age > 65 THEN 'Pensioner'
            ELSE 'Unknown'
        END AS age_group,

        CASE
            WHEN Race IS NULL
              OR TRIM(Race) = ''
              OR Race = 'other'
            THEN 'None'
            ELSE Race
        END AS Race,

        CASE
            WHEN Gender IS NULL
              OR TRIM(Gender) = ''
            THEN 'None'
            ELSE Gender
        END AS Gender

    FROM retail.default.userprofiles
)

SELECT *
FROM user_profiles;


WITH viewership AS (

    SELECT

        COALESCE(UserID0, UserID4) AS UserID,

        TO_CHAR(RecordDate2, 'yyyyMM') AS month_id,

        TO_DATE(RecordDate2) AS watch_date,

        DAYNAME(RecordDate2) AS day_name,

        CASE
            WHEN DAYNAME(RecordDate2) IN ('Sat', 'Sun')
            THEN 'Weekend'
            ELSE 'Weekday'
        END AS day_classification,

        MONTHNAME(RecordDate2) AS month_name,

        CASE
            WHEN Channel2 IN ('SawSee', 'Sawsee')
            THEN 'SawSee'

            WHEN Channel2 IN (
                'SuperSport Live Events',
                'Live on SuperSport',
                'Supersport Live Events',
                'DStv Events 1'
            )
            THEN 'Live Events'

            ELSE Channel2
        END AS TV_channel,

        DATE_FORMAT(RecordDate2, 'HH:mm:ss') AS watch_time,

        CASE
            WHEN HOUR(RecordDate2) BETWEEN 0 AND 5
                THEN 'Midnight'

            WHEN HOUR(RecordDate2) BETWEEN 6 AND 11
                THEN 'Morning'

            WHEN HOUR(RecordDate2) BETWEEN 12 AND 16
                THEN 'Afternoon'

            ELSE 'Evening'
        END AS time_of_day,

        HOUR(RecordDate2) AS hour_of_day,

        DATE_FORMAT(`Duration 2`, 'HH:mm:ss') AS duration

    FROM retail.default.viewership
)

SELECT *
FROM viewership;


--joining tables
CREATE OR REPLACE TEMP VIEW final_viewership AS
WITH user_profiles AS (

    SELECT
        UserID,

        CASE
            WHEN Province IS NULL
              OR TRIM(Province) = ''
              OR Province = 'None'
            THEN 'Uncategorized'
            ELSE Province
        END AS Region,

        CASE
            WHEN age = 0 THEN 'Infants'
            WHEN age BETWEEN 1 AND 12 THEN 'Kids'
            WHEN age BETWEEN 13 AND 19 THEN 'Teenager'
            WHEN age BETWEEN 20 AND 35 THEN 'Youth'
            WHEN age BETWEEN 36 AND 50 THEN 'Adult'
            WHEN age BETWEEN 51 AND 65 THEN 'Elder'
            WHEN age > 65 THEN 'Pensioner'
            ELSE 'Unknown'
        END AS age_group,

        Gender,
        Race

    FROM retail.default.userprofiles
),

viewership AS (

    SELECT

        COALESCE(UserID0, UserID4) AS UserID,

        TO_DATE(RecordDate2) AS watch_date,

        MONTHNAME(RecordDate2) AS month_name,

        DAYNAME(RecordDate2) AS day_name,

        CASE
            WHEN DAYNAME(RecordDate2) IN ('Sat','Sun')
            THEN 'Weekend'
            ELSE 'Weekday'
        END AS day_classification,

        CASE
            WHEN Channel2 IN ('SawSee','Sawsee')
            THEN 'SawSee'

            WHEN Channel2 IN (
                'SuperSport Live Events',
                'Live on SuperSport',
                'Supersport Live Events',
                'DStv Events 1'
            )
            THEN 'Live Events'

            ELSE Channel2
        END AS TV_channel,

        HOUR(RecordDate2) AS hour_of_day,

        `Duration 2` AS duration,

        CASE
            WHEN HOUR(RecordDate2) BETWEEN 0 AND 5
                THEN 'Midnight'
            WHEN HOUR(RecordDate2) BETWEEN 6 AND 11
                THEN 'Morning'
            WHEN HOUR(RecordDate2) BETWEEN 12 AND 16
                THEN 'Afternoon'
            ELSE 'Evening'
        END AS time_of_day,

        CASE
            WHEN HOUR(`Duration 2`) * 60 + MINUTE(`Duration 2`) < 30 THEN 'Short'
            WHEN HOUR(`Duration 2`) * 60 + MINUTE(`Duration 2`) <= 60 THEN 'Medium'
            ELSE 'Long'
        END AS screen_time_bucket

    FROM retail.default.viewership
)

SELECT

    v.UserID AS subscriber_id,
    v.watch_date,
    v.month_name,
    v.day_name,
    v.day_classification,
    v.TV_channel,
    v.hour_of_day,
    v.duration,
    v.time_of_day,
    v.screen_time_bucket,

    u.Region,
    u.age_group,
    u.Gender,
    u.Race

FROM viewership v

LEFT JOIN user_profiles u
    ON v.UserID = u.UserID;

    --checking how big is the audience
    SELECT
    COUNT(*) AS viewing_records,
    COUNT(DISTINCT subscriber_id) AS unique_viewers
FROM final_viewership;

--checking which group are active
SELECT
    age_group,
    COUNT(*) AS viewing_records,
    COUNT(DISTINCT subscriber_id) AS unique_viewers
FROM final_viewership
GROUP BY age_group
ORDER BY viewing_records DESC;

--checking the location of viewers
SELECT
    Region,
    COUNT(DISTINCT subscriber_id) AS unique_viewers,
    COUNT(*) AS viewing_records
FROM final_viewership
GROUP BY Region
ORDER BY viewing_records DESC;

--checking weekday vs weekend
SELECT
    day_classification,
    COUNT(*) AS viewing_records,
    COUNT(DISTINCT subscriber_id) AS unique_viewers
FROM final_viewership
GROUP BY day_classification
ORDER BY viewing_records DESC;

--checking time of day
SELECT
    time_of_day,
    COUNT(*) AS viewing_records
FROM final_viewership
GROUP BY time_of_day
ORDER BY viewing_records DESC;

--checking hourly viewing patterns
SELECT
    hour_of_day,
    COUNT(*) AS viewing_records
FROM final_viewership
GROUP BY hour_of_day
ORDER BY hour_of_day;

--checking channel performance
SELECT
    TV_channel,
    COUNT(*) AS viewing_records,
    COUNT(DISTINCT subscriber_id) AS unique_viewers
FROM final_viewership
GROUP BY TV_channel
ORDER BY viewing_records DESC;

--checking creen time
SELECT
    screen_time_bucket,
    COUNT(*) AS viewing_records,
    COUNT(DISTINCT subscriber_id) AS unique_viewers
FROM final_viewership
GROUP BY screen_time_bucket
ORDER BY viewing_records DESC;

--checking mointhly trends
SELECT
    DATE_FORMAT(watch_date, 'yyyy-MM') AS year_month,
    COUNT(*) AS viewing_records,
    COUNT(DISTINCT subscriber_id) AS unique_viewers
FROM final_viewership
GROUP BY DATE_FORMAT(watch_date, 'yyyy-MM')
ORDER BY year_month;


--checking the KPI's
SELECT

    COUNT(*) AS total_viewing_records,

    COUNT(DISTINCT subscriber_id) AS unique_viewers,

    COUNT(DISTINCT TV_channel) AS channels,

    COUNT(DISTINCT Region) AS regions,

    COUNT(DISTINCT watch_date) AS active_days

FROM final_viewership;